# HW0: from NumPy to language models

**RLHF course · Pegah Alizadeh** · **Due before session 2** · About 1.5 hours · No GPU needed

The session 1 lab used **PyTorch** and a real language model. This notebook goes over the same tools in small steps, starting from NumPy, which you already know. Do it before HW1: the next labs build on it.

| Part | What you learn | Exercises |
|---|---|---|
| 1 | PyTorch tensors, compared with NumPy arrays | 1 |
| 2 | Working with batches of sequences: shapes, `gather`, shifting and masking | 2, 3 |
| 3 | Automatic gradients | none |
| 4 | A real language model: tokens, logits, log-probabilities, sampling, chat templates | 4, 5 |

Every exercise has a check: run it after your answer. If it fails, read the error, try again, and then open the solution if you need it. **Exercise 3 is the most important one: it's Exercise B of the lab, on tiny tensors.**

To run it: open it in Google Colab, then use **Runtime → Run all**. A CPU is enough.

In [ ]:
%pip install -q -U transformers

---
## Part 1: tensors are like NumPy arrays

A PyTorch **tensor** is an array, like a NumPy array, with two extras: it can run on a GPU, and it can compute gradients automatically. Most operations have the same name.

| NumPy | PyTorch |
|---|---|
| `np.array([...])` | `torch.tensor([...])` |
| `a.shape` | `t.shape` |
| `a.sum(axis=1)` | `t.sum(dim=1)` |
| `np.exp(a)`, `np.log(a)` | `torch.exp(t)`, `torch.log(t)` |
| `a[:, :-1]`, `a[:, 1:]` | the same |
| `a[:, None]` | `t.unsqueeze(1)` |

In [ ]:
import numpy as np
import torch

a = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
t = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])

print("shapes:", a.shape, t.shape)
print("sum over each row:", a.sum(axis=1), t.sum(dim=1))
print("drop the last column:\n", t[:, :-1])
print("back to NumPy:", t.numpy().dtype, "and from NumPy:", torch.from_numpy(a).dtype)

### Exercise 1: softmax

Softmax turns scores into probabilities. Here is a numerically stable NumPy version: subtracting the maximum doesn't change the result, but it prevents `exp` from overflowing. Write the same function in PyTorch.

<details><summary>Solution</summary>

```python
def softmax_torch(z):
    z = z - z.max()
    e = torch.exp(z)
    return e / e.sum()
```
</details>

In [ ]:
def softmax_numpy(z):
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()

In [ ]:
def softmax_torch(z):
    # TODO: the same three steps as the NumPy version above, with torch functions
    raise NotImplementedError("Complete exercise 1")

In [ ]:
# Check
z = torch.tensor([2.0, 1.0, 0.1, 1000.0])
assert torch.allclose(softmax_torch(z), torch.softmax(z, dim=0))
assert np.allclose(softmax_torch(z).numpy(), softmax_numpy(z.numpy()))
print("Exercise 1: correct. PyTorch also has it built in: torch.softmax(z, dim=0)")

---
## Part 2: batches of sequences

A language model works on a **batch** of $B$ sequences, each of length $L$, and at every position it gives one score for each of the $V$ tokens in its vocabulary. Its output therefore has shape `(B, L, V)`.

Here is a tiny made-up example: 2 sequences, 4 positions, and a vocabulary of 5 tokens. `log_softmax` over the last dimension turns each row of scores into log-probabilities.

In [ ]:
torch.manual_seed(0)
logits = torch.randn(2, 4, 5)                  # (B, L, V)
log_probs = logits.log_softmax(dim=-1)         # still (B, L, V); each row of V values now sums to 1 after exp
print(log_probs.shape)
print("probabilities at sequence 0, position 0 sum to:", log_probs[0, 0].exp().sum().item())

### Exercise 2: pick out one value per position with `gather`

At each position we want the log-probability of **one** particular token, given in `tokens`, of shape `(2, 4)`. The slow, clear way is a double loop, shown below. Write the fast version with `gather`.

`gather(-1, index)` reads values along the last dimension. The index must have the same number of dimensions as `log_probs`, so add one with `unsqueeze(-1)`, then remove it afterwards with `squeeze(-1)`.

<details><summary>Solution</summary>

```python
picked_fast = log_probs.gather(-1, tokens.unsqueeze(-1)).squeeze(-1)
```
</details>

In [ ]:
tokens = torch.tensor([[1, 0, 3, 2], [4, 4, 0, 1]])   # (B, L): one token id per position

picked_slow = torch.zeros(2, 4)
for b in range(2):
    for position in range(4):
        picked_slow[b, position] = log_probs[b, position, tokens[b, position]]
print(picked_slow)

In [ ]:
# TODO: one line, with log_probs.gather(...). The result should have shape (2, 4).
picked_fast = None

In [ ]:
# Check
assert picked_fast is not None, "Complete exercise 2"
assert picked_fast.shape == (2, 4), f"Expected shape (2, 4), got {tuple(picked_fast.shape)}"
assert torch.allclose(picked_fast, picked_slow)
print("Exercise 2: correct")

### Exercise 3: the log-probability of a sequence

This is the most important exercise. It's the lab's Exercise B, on a tiny example.

A language model reads tokens $x_1, \ldots, x_L$, and **its scores at position $t$ predict the token at position $t+1$**. So the scores and the tokens are offset by one:

- **Scores:** use positions $1$ to $L-1$, so drop the last one with `logits[:, :-1]`. The last position predicts a token we don't have.
- **Tokens to score:** use positions $2$ to $L$, so drop the first one with `input_ids[:, 1:]`.

We also want to count only some positions: in instruction tuning, only the response, not the prompt. A **mask** of 0s and 1s says which tokens count. It is shifted the same way as the tokens.

The slow, clear version is below. Write the fast version without loops.

<details><summary>Solution</summary>

```python
def sequence_log_prob(logits, input_ids, mask):
    log_probs = logits[:, :-1].log_softmax(dim=-1)
    targets = input_ids[:, 1:]
    token_log_probs = log_probs.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
    return (token_log_probs * mask[:, 1:]).sum(dim=-1)
```
</details>

In [ ]:
def sequence_log_prob_slow(logits, input_ids, mask):
    B, L, V = logits.shape
    totals = torch.zeros(B)
    for b in range(B):
        for t in range(L - 1):                      # the scores at position t predict token t + 1
            if mask[b, t + 1] == 1:                  # count only the tokens the mask selects
                scores = logits[b, t]
                totals[b] += scores.log_softmax(dim=-1)[input_ids[b, t + 1]]
    return totals


input_ids = torch.tensor([[3, 1, 4, 1], [2, 0, 4, 3]])
mask = torch.tensor([[0, 0, 1, 1], [0, 1, 1, 0]])     # 1 = this token counts
print(sequence_log_prob_slow(logits, input_ids, mask))

In [ ]:
def sequence_log_prob(logits, input_ids, mask):
    # TODO: shift, apply log_softmax, gather, multiply by the mask, sum over positions.
    # Return a tensor of shape (B,).
    raise NotImplementedError("Complete exercise 3")

In [ ]:
# Check
fast = sequence_log_prob(logits, input_ids, mask)
assert fast.shape == (2,), f"Expected shape (2,), got {tuple(fast.shape)}"
assert torch.allclose(fast, sequence_log_prob_slow(logits, input_ids, mask)), "Check the shift and the mask."
print("Exercise 3: correct")

---
## Part 3: automatic gradients

PyTorch can compute gradients for you. Mark a tensor with `requires_grad=True`, compute something from it, and call `.backward()`.

Here is a loss you'll meet in session 2: $\ell(\Delta) = -\log \sigma(\Delta)$. Its derivative is $-\sigma(-\Delta)$, so we can check PyTorch's answer by hand.

In [ ]:
delta = torch.tensor(0.3, requires_grad=True)
loss = -torch.nn.functional.logsigmoid(delta)
loss.backward()
print("PyTorch:  ", delta.grad.item())
print("By hand:  ", -torch.sigmoid(-delta).item())

Training a model repeats three steps: compute a loss, call `.backward()`, then move every parameter a little against its gradient. The lab's training loop does exactly this, so you won't need to write it yourself.

---
## Part 4: a real language model

We use GPT-2, a small, older model that runs on a CPU. It's a **base model**: trained to continue text, not to answer questions.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").eval()

text = "The capital of France is"
ids = tokenizer(text, return_tensors="pt")["input_ids"]
print("token ids:", ids)
print("tokens:   ", [tokenizer.decode(i) for i in ids[0]])

with torch.no_grad():
    logits = model(ids).logits
print("logits shape (B, L, V):", logits.shape)

**Try it:** tokenize a long or unusual word, and your own name. How many tokens does each one become?

### Exercise 4: the most likely next tokens

Use the scores at the **last position**, `logits[0, -1]`, apply softmax, and print the 5 most likely next tokens with their probabilities. `torch.topk` gives the largest values and their positions.

<details><summary>Solution</summary>

```python
probs = logits[0, -1].softmax(dim=-1)
top = torch.topk(probs, 5)
for p, i in zip(top.values, top.indices):
    print(f"{tokenizer.decode(i)!r}: {p:.3f}")
```
</details>

In [ ]:
# TODO: the 5 most likely next tokens after "The capital of France is", with their probabilities

### Exercise 5: compare two sentences

Use your `sequence_log_prob` from exercise 3, with a mask of all 1s, to compute the log-probability of each sentence below. Which one does the model find more likely? Is the difference large?

The result is the log-probability of everything after the first token, given the first token.

In [ ]:
for sentence in ["The capital of France is Paris.", "The capital of France is Berlin."]:
    ids = tokenizer(sentence, return_tensors="pt")["input_ids"]
    with torch.no_grad():
        logits = model(ids).logits
    score = sequence_log_prob(logits, ids, torch.ones_like(ids))
    print(f"{score.item():8.2f}  {sentence}")

### Sampling and temperature

A model generates text by repeating one step: predict, pick a token, append it. **Temperature** divides the scores before softmax. A low temperature concentrates on likely tokens, and a high one spreads out.

Run this with temperatures 0.3, 1.0 and 2.0. What changes? Which would you trust for a factual answer?

In [ ]:
temperature = 1.0
ids = tokenizer("The capital of France is", return_tensors="pt")["input_ids"]
torch.manual_seed(0)
out = model.generate(ids, max_new_tokens=30, do_sample=True, temperature=temperature, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0]))

### Chat templates

GPT-2 has no idea what a conversation is. Instruction-tuned models use a **chat template**, which turns a list of messages into one flat text. Here is the template of the model we use in the lab.

In [ ]:
chat_tokenizer = AutoTokenizer.from_pretrained("allenai/OLMo-2-0425-1B-SFT")
messages = [
    {"role": "user", "content": "What is the capital of France?"},
    {"role": "assistant", "content": "The capital of France is Paris."},
]
print(chat_tokenizer.apply_chat_template(messages, tokenize=False))
print("---")
print(chat_tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True))

The second text stops exactly where the model should start its answer. The difference between the two is the response, and in instruction tuning those are the only tokens with a 1 in the mask, as in exercise 3.

**You now have the tools for the next labs.**

**Sources.** Exercise 3 follows the tensor walk-through in N. Lambert's Lecture 0, "The ML foundations of LLM post-training" ([rlhfbook.com/course](https://rlhfbook.com/course)). For chat templates, see Chapter 4 of the RLHF book and the [Hugging Face LLM course](https://huggingface.co/learn/llm-course), which is also good further reading.